# 02 - Diagnóstico y reconstrucción de archivos Parquet

## Fase 2: Diagnóstico y reconstrucción

En esta fase se revisan los esquemas reales de los archivos Parquet leídos en la Fase 1 y se reconstruye un dataset operativo bajo un **esquema canónico común**.

La idea es sencilla:

1. Revisar cómo viene cada archivo.
2. Comparar su esquema contra un esquema esperado.
3. Detectar columnas faltantes, columnas adicionales y tipos incompatibles.
4. Homologar columnas de `yellow`, `green` y `fhvhv`.
5. Agregar columnas ausentes con valores nulos controlados.
6. Convertir tipos de datos.
7. Separar archivos recuperables y no recuperables.
8. Guardar el resultado reconstruido en la capa `bronze`.

## 1. Objetivo de la fase

El objetivo de esta fase no es limpiar todavía los datos de negocio.

El objetivo es que todos los servicios tengan la misma estructura.

Ejemplo:

- `yellow` usa `tpep_pickup_datetime`
- `green` usa `lpep_pickup_datetime`
- `fhvhv` usa `pickup_datetime`

Pero los tres significan lo mismo:

`pickup_datetime`

Por eso se necesita una matriz de homologación.

In [1]:
# Configuración inicial
# Esta celda prepara Python, Hadoop y Spark para trabajar en Windows.

import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime
from functools import reduce

# Java 17 usado por Spark.
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = r"C:\Program Files\Java\jdk-17\bin;" + os.environ["PATH"]

# Hadoop en Windows.
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

# Python usado por Spark.
PYTHON_PATH = sys.executable
os.environ["PYSPARK_PYTHON"] = PYTHON_PATH
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_PATH

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType,
    TimestampType
)

spark = (
    SparkSession.builder
    .appName("NYC_TAXI_ETL_02_DIAGNOSTICO_RECONSTRUCCION")
    .master("local[*]")
    .config("spark.pyspark.python", PYTHON_PATH)
    .config("spark.pyspark.driver.python", PYTHON_PATH)
    .config("spark.sql.session.timeZone", "America/Guayaquil")
    .config("spark.hadoop.io.native.lib.available", "false")
    .config("spark.sql.files.ignoreCorruptFiles", "false")
    .config("spark.sql.files.ignoreMissingFiles", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)
print("Python usado por Spark:", PYTHON_PATH)
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("winutils existe:", Path(r"C:\hadoop\bin\winutils.exe").exists())

Spark iniciado correctamente
Versión de Spark: 4.1.2
Python usado por Spark: c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
HADOOP_HOME: C:\hadoop
winutils existe: False


In [2]:
# Rutas del proyecto
# El notebook está dentro de la carpeta notebooks, por eso se usa ".." para volver a la raíz.

PROJECT_PATH = Path.cwd().parent

DATA_PATH = PROJECT_PATH / "data"
RAW_PATH = DATA_PATH / "raw"
BRONZE_PATH = DATA_PATH / "bronze"
QUARANTINE_PATH = DATA_PATH / "quarantine"
AUDIT_PATH = DATA_PATH / "audit"
METADATA_PATH = PROJECT_PATH / "metadata"
CONFIG_PATH = PROJECT_PATH / "config"

for path in [BRONZE_PATH, QUARANTINE_PATH, AUDIT_PATH, METADATA_PATH, CONFIG_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Ruta del proyecto:", PROJECT_PATH)
print("Ruta raw:", RAW_PATH)
print("Ruta bronze:", BRONZE_PATH)

Ruta del proyecto: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
Ruta raw: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw
Ruta bronze: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze


In [3]:
# Crear process_id de la Fase 2.
# Este identificador permite rastrear cada ejecución del pipeline.

PROCESS_ID = "fase2_" + datetime.now().strftime("%Y%m%d_%H%M%S")

print("PROCESS_ID:", PROCESS_ID)

PROCESS_ID: fase2_20260617_170931


## 2. Cargar `audit_file_inventory`

Se carga el inventario generado en la Fase 1.

Ese inventario indica qué archivos se pudieron leer y cuáles fallaron.  
Para la Fase 2 se usan principalmente los archivos con `READ_OK`.

Si no se encuentra el inventario, el notebook puede reconstruir una lista de archivos desde la carpeta `raw`.

In [4]:
# Función auxiliar para buscar el último inventario guardado en audit.

def find_latest_audit_inventory(audit_path: Path):
    candidates = []

    for item in audit_path.iterdir():
        if item.is_dir() and item.name.startswith("audit_file_inventory_"):
            candidates.append(item)

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

latest_audit_path = find_latest_audit_inventory(AUDIT_PATH)

print("Último inventario encontrado:", latest_audit_path)

Último inventario encontrado: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\audit_file_inventory_2e9a7366-d209-4985-ad78-8c8ca2a14476


In [5]:
# Cargar audit_file_inventory.
# Si existe el inventario en Parquet, se usa.
# Si no existe, se reconstruye una lista básica desde raw para poder continuar.

def infer_service_type(file_path: Path):
    parts = [p.lower() for p in file_path.parts]
    if "yellow" in parts:
        return "yellow"
    if "green" in parts:
        return "green"
    if "fhvhv" in parts:
        return "fhvhv"
    if "bad_parquet" in parts:
        return "bad_parquet"
    return "unknown"

def infer_year_month(file_path: Path):
    year = None
    month = None

    for part in file_path.parts:
        if part.startswith("year="):
            year = part.replace("year=", "")
        if part.startswith("month="):
            month = part.replace("month=", "")

    return year, month

if latest_audit_path is not None:
    audit_file_inventory = spark.read.parquet(str(latest_audit_path))
    print("audit_file_inventory cargado desde Parquet.")
else:
    print("No se encontró audit_file_inventory. Se reconstruye desde raw.")

    rows = []
    for file in RAW_PATH.rglob("*.parquet"):
        year, month = infer_year_month(file)
        rows.append({
            "process_id": PROCESS_ID,
            "source_system": "NYC_TLC_OR_APACHE_TESTING",
            "service_type": infer_service_type(file),
            "file_name": file.name,
            "file_path": str(file),
            "file_size_mb": round(file.stat().st_size / (1024 * 1024), 4),
            "partition_year": year,
            "partition_month": month,
            "read_status": "PENDING",
            "record_count": None,
            "column_count": None,
            "schema_hash": None,
            "error_message": None,
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    audit_file_inventory = spark.createDataFrame(rows)

audit_file_inventory.show(20, truncate=False)

audit_file_inventory cargado desde Parquet.
+------------------------------------+----------------------+------------+-----------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------+---------------+-----------+------------+------------+--------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
# Seleccionar archivos leídos correctamente.
# Los archivos READ_ERROR no se reconstruyen; se envían a cuarentena técnica.

readable_files_df = (
    audit_file_inventory
    .filter(F.col("service_type").isin("yellow", "green", "fhvhv"))
    .filter((F.col("read_status") == "READ_OK") | (F.col("read_status") == "PENDING"))
)

readable_files = readable_files_df.collect()

print("Archivos candidatos para reconstrucción:", len(readable_files))

for row in readable_files:
    print(row["service_type"], "-", row["file_name"])

Archivos candidatos para reconstrucción: 11
yellow - yellow_tripdata_2020-01.parquet
yellow - yellow_tripdata_2022-01.parquet
yellow - yellow_tripdata_2022-02.parquet
yellow - yellow_tripdata_2023-01.parquet
yellow - yellow_tripdata_2023-03.parquet
yellow - yellow_tripdata_2021-01.parquet
yellow - yellow_tripdata_2023-02.parquet
yellow - yellow_tripdata_2023-04.parquet
fhvhv - fhvhv_tripdata_2023-01.parquet
green - green_tripdata_2023-01.parquet
green - green_tripdata_2023-02.parquet


## Tratamiento de archivos `bad_parquet` con estado `READ_OK`

Durante la Fase 1 se analizaron los archivos de prueba provenientes de `APACHE_PARQUET_TESTING`. Como resultado, algunos archivos presentaron errores reales de lectura (`READ_ERROR`) y fueron enviados a cuarentena técnica. Sin embargo, otros archivos pudieron ser leídos correctamente por Spark y quedaron registrados como `READ_OK` en `audit_file_inventory`.

Aunque estos archivos son técnicamente legibles, no corresponden a fuentes de negocio del caso de estudio, ya que no pertenecen a los servicios `yellow`, `green` o `fhvhv`. Por este motivo, no participan en el proceso de homologación ni en la construcción del esquema canónico de viajes de la Fase 2.

Estos archivos se consideran casos de prueba utilizados para validar la robustez del proceso de lectura y diagnóstico. Los `READ_ERROR` permanecen en la cuarentena técnica definida en la Fase 1. Los `READ_OK` se documentan como excluidos en la Fase 2 porque no pertenecen a una fuente de negocio ni al esquema canónico de viajes. Su exclusión es una decisión funcional del proyecto y no un error de procesamiento.


## 3. Construir esquema esperado

Se define un esquema esperado mínimo por tipo de servicio.

Este esquema no busca tener todas las columnas posibles, sino las columnas necesarias para reconstruir el esquema canónico.

In [7]:
# Esquemas esperados por servicio.
# Representan las columnas necesarias para homologar cada fuente.

EXPECTED_SCHEMAS = {
    "yellow": {
        "VendorID": "numeric",
        "tpep_pickup_datetime": "timestamp",
        "tpep_dropoff_datetime": "timestamp",
        "passenger_count": "numeric",
        "trip_distance": "numeric",
        "PULocationID": "numeric",
        "DOLocationID": "numeric",
        "payment_type": "numeric",
        "fare_amount": "numeric",
        "extra": "numeric",
        "mta_tax": "numeric",
        "tip_amount": "numeric",
        "tolls_amount": "numeric",
        "total_amount": "numeric",
        "congestion_surcharge": "numeric",
        "airport_fee": "numeric"
    },
    "green": {
        "VendorID": "numeric",
        "lpep_pickup_datetime": "timestamp",
        "lpep_dropoff_datetime": "timestamp",
        "passenger_count": "numeric",
        "trip_distance": "numeric",
        "PULocationID": "numeric",
        "DOLocationID": "numeric",
        "payment_type": "numeric",
        "fare_amount": "numeric",
        "extra": "numeric",
        "mta_tax": "numeric",
        "tip_amount": "numeric",
        "tolls_amount": "numeric",
        "total_amount": "numeric",
        "congestion_surcharge": "numeric"
    },
    "fhvhv": {
        "pickup_datetime": "timestamp",
        "dropoff_datetime": "timestamp",
        "PULocationID": "numeric",
        "DOLocationID": "numeric",
        "trip_miles": "numeric",
        "base_passenger_fare": "numeric",
        "tolls": "numeric",
        "sales_tax": "numeric",
        "tips": "numeric"
    }
}

# Guardar esquemas esperados como evidencia en metadata.
for service, schema in EXPECTED_SCHEMAS.items():
    output_file = METADATA_PATH / f"expected_schema_{service}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(schema, f, indent=4, ensure_ascii=False)

print("Esquemas esperados guardados en metadata.")

Esquemas esperados guardados en metadata.


## 4. Comparar esquemas reales

Se lee cada archivo candidato y se obtiene su esquema real.

Con esto se puede revisar qué columnas trae realmente cada archivo.

In [8]:
# Funciones auxiliares para diagnóstico de esquemas.

def normalize_type(data_type):
    # Agrupa tipos de Spark en categorías simples para comparar.
    dtype = data_type.simpleString().lower()

    if "timestamp" in dtype or "date" in dtype:
        return "timestamp"
    if any(t in dtype for t in ["int", "bigint", "long", "double", "float", "decimal"]):
        return "numeric"
    if "string" in dtype:
        return "string"
    if "boolean" in dtype:
        return "boolean"

    return dtype

def get_schema_hash(schema):
    # Genera hash del esquema para identificar estructuras iguales o diferentes.
    schema_text = schema.simpleString()
    return hashlib.md5(schema_text.encode("utf-8")).hexdigest()

def clean_error_message(error):
    # Reduce errores largos para guardarlos en tablas de auditoría.
    message = str(error).replace("\n", " ").replace("\r", " ")
    return message[:1000]

def read_file_schema(file_path):
    # Lee un archivo Parquet y devuelve DataFrame + metadatos de esquema.
    df = spark.read.parquet(str(file_path))
    actual_schema = {field.name: normalize_type(field.dataType) for field in df.schema.fields}
    return df, actual_schema, get_schema_hash(df.schema)

In [9]:
# Diagnóstico de esquemas por archivo.
# Compara el esquema real contra el esquema esperado.
# Se define un schema explícito para evitar errores cuando hay valores vacíos.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema_diagnosis_rows = []

for row in readable_files:
    service_type = row["service_type"]
    file_name = row["file_name"]
    file_path = Path(row["file_path"])
    expected_schema = EXPECTED_SCHEMAS.get(service_type, {})

    try:
        df_tmp, actual_schema, schema_hash = read_file_schema(file_path)

        actual_columns = set(actual_schema.keys())
        expected_columns = set(expected_schema.keys())

        missing_columns = sorted(list(expected_columns - actual_columns))
        additional_columns = sorted(list(actual_columns - expected_columns))

        incompatible_types = []

        for col_name, expected_type in expected_schema.items():
            if col_name in actual_schema and actual_schema[col_name] != expected_type:
                incompatible_types.append(
                    f"{col_name}: esperado={expected_type}, real={actual_schema[col_name]}"
                )

        if len(missing_columns) == 0 and len(incompatible_types) == 0:
            reconstruction_status = "RECUPERABLE_OK"
        elif len(missing_columns) > 0 and len(incompatible_types) == 0:
            reconstruction_status = "RECUPERABLE_MISSING_COLUMNS"
        elif len(incompatible_types) > 0:
            reconstruction_status = "RECUPERABLE_TYPE_CASTING"
        else:
            reconstruction_status = "PARTIALLY_RECOVERABLE"

        schema_diagnosis_rows.append({
            "process_id": str(PROCESS_ID),
            "service_type": str(service_type),
            "file_name": str(file_name),
            "file_path": str(file_path),
            "schema_hash": str(schema_hash),
            "actual_column_count": int(len(actual_columns)),
            "expected_column_count": int(len(expected_columns)),
            "missing_columns": ", ".join(missing_columns),
            "additional_columns": ", ".join(additional_columns),
            "incompatible_types": ", ".join(incompatible_types),
            "reconstruction_status": str(reconstruction_status),
            "error_message": "",
            "diagnosed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

    except Exception as e:
        schema_diagnosis_rows.append({
            "process_id": str(PROCESS_ID),
            "service_type": str(service_type),
            "file_name": str(file_name),
            "file_path": str(file_path),
            "schema_hash": "",
            "actual_column_count": 0,
            "expected_column_count": int(len(expected_schema)),
            "missing_columns": "",
            "additional_columns": "",
            "incompatible_types": "",
            "reconstruction_status": "NOT_RECOVERABLE_CORRUPT_METADATA",
            "error_message": clean_error_message(e),
            "diagnosed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

schema_diagnosis_schema = StructType([
    StructField("process_id", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("schema_hash", StringType(), True),
    StructField("actual_column_count", IntegerType(), True),
    StructField("expected_column_count", IntegerType(), True),
    StructField("missing_columns", StringType(), True),
    StructField("additional_columns", StringType(), True),
    StructField("incompatible_types", StringType(), True),
    StructField("reconstruction_status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("diagnosed_at", StringType(), True)
])

schema_diagnosis_df = spark.createDataFrame(
    schema_diagnosis_rows,
    schema=schema_diagnosis_schema
)

schema_diagnosis_df.select(
    "service_type",
    "file_name",
    "actual_column_count",
    "missing_columns",
    "additional_columns",
    "incompatible_types",
    "reconstruction_status"
).show(50, truncate=False)

+------------+-------------------------------+-------------------+---------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------------------------+
|service_type|file_name                      |actual_column_count|missing_columns|additional_columns                                                                                                                                                                                                                                             |incompatible_types|reconstruction_status      |
+------------+-------------------------------+-------------------+---------------+--------------------------------------------------------------------------------------------------------------------------------------------------

## 5. Detectar columnas faltantes

Las columnas faltantes son columnas esperadas que no aparecen en un archivo.

Esto no siempre significa que el archivo está dañado.  
A veces simplemente significa que un servicio no maneja ese campo.

In [10]:
# Archivos con columnas faltantes.

missing_columns_df = (
    schema_diagnosis_df
    .filter(F.col("missing_columns").isNotNull())
    .filter(F.length(F.col("missing_columns")) > 0)
    .select("service_type", "file_name", "missing_columns", "reconstruction_status")
)

missing_columns_df.show(50, truncate=False)

+------------+-------------------------------+---------------+---------------------------+
|service_type|file_name                      |missing_columns|reconstruction_status      |
+------------+-------------------------------+---------------+---------------------------+
|yellow      |yellow_tripdata_2023-03.parquet|airport_fee    |RECUPERABLE_MISSING_COLUMNS|
|yellow      |yellow_tripdata_2023-02.parquet|airport_fee    |RECUPERABLE_MISSING_COLUMNS|
|yellow      |yellow_tripdata_2023-04.parquet|airport_fee    |RECUPERABLE_MISSING_COLUMNS|
+------------+-------------------------------+---------------+---------------------------+



### Interpretación de columnas faltantes

Esta tabla muestra si a un archivo le falta alguna columna nativa esperada para su propio servicio.

Si la tabla aparece vacía, el resultado es positivo: significa que `EXPECTED_SCHEMAS` fue definido con los nombres reales de cada fuente y que, dentro de ese servicio, la estructura se mantiene estable.

Si aparecen filas, esas columnas faltantes no detienen el pipeline. En la reconstrucción se completan como valores nulos controlados mediante `safe_col` y `add_missing_canonical_columns`, para conservar el esquema canónico sin perder trazabilidad.

## 6. Detectar columnas adicionales

Las columnas adicionales son columnas que existen en el archivo, pero no están dentro del esquema esperado mínimo.

No necesariamente son malas.  
Solo se registran para auditoría.

In [11]:
# Archivos con columnas adicionales.

additional_columns_df = (
    schema_diagnosis_df
    .filter(F.col("additional_columns").isNotNull())
    .filter(F.length(F.col("additional_columns")) > 0)
    .select("service_type", "file_name", "additional_columns")
)

additional_columns_df.show(50, truncate=False)

+------------+-------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|service_type|file_name                      |additional_columns                                                                                                                                                                                                                                             |
+------------+-------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|yellow      |yellow_tripdata_2020-01.parquet|RatecodeID, improvement_surcharge, store_and_

### Interpretación de columnas adicionales

Esta tabla muestra columnas que existen en el archivo real, pero no forman parte del esquema esperado usado para esta fase.

Si la tabla aparece vacía, no se detectaron columnas adicionales no documentadas frente a `EXPECTED_SCHEMAS`. Si contiene filas, cada nombre listado es evidencia de una diferencia real encontrada por Spark.

Las columnas adicionales no se incorporan automáticamente al esquema canónico porque pueden ser específicas de un servicio o no aportar directamente al análisis unificado de viajes. Se conservan en `additional_columns_df` como evidencia técnica y podrían evaluarse como enriquecimiento futuro si el análisis de negocio lo requiere.

## 7. Detectar tipos incompatibles

Un tipo incompatible aparece cuando una columna existe, pero Spark la lee con un tipo diferente al esperado.

En esta fase no se elimina el archivo.  
Primero se intenta recuperar mediante conversión controlada.

In [12]:
# Archivos con tipos incompatibles.

incompatible_types_df = (
    schema_diagnosis_df
    .filter(F.col("incompatible_types").isNotNull())
    .filter(F.length(F.col("incompatible_types")) > 0)
    .select("service_type", "file_name", "incompatible_types", "reconstruction_status")
)

incompatible_types_df.show(50, truncate=False)

+------------+---------+------------------+---------------------+
|service_type|file_name|incompatible_types|reconstruction_status|
+------------+---------+------------------+---------------------+
+------------+---------+------------------+---------------------+



### Interpretación de tipos incompatibles

Esta tabla permite verificar si los tipos reales de las columnas coinciden con los tipos definidos en `EXPECTED_SCHEMAS`.

Si aparece vacía, significa que los tipos esperados ya coinciden con los tipos reales observados en los archivos. Si aparecen incompatibilidades, se resuelven durante la homologación mediante conversiones controladas con `safe_col`, evitando que un cambio de tipo detenga el proceso completo.

## 8. Construir matriz de homologación

Esta matriz explica cómo se convierten los nombres reales de cada servicio al esquema canónico.

Es importante justificar esto porque `yellow`, `green` y `fhvhv` usan nombres diferentes para conceptos parecidos.

In [13]:
# Matriz de homologación.
# Esta tabla se muestra en el notebook y se guarda como evidencia.

homologation_matrix = [
    {
        "concepto_canonico": "pickup_datetime",
        "yellow": "tpep_pickup_datetime",
        "green": "lpep_pickup_datetime",
        "fhvhv": "pickup_datetime",
        "justificacion": "Representa la fecha y hora de recogida del viaje."
    },
    {
        "concepto_canonico": "dropoff_datetime",
        "yellow": "tpep_dropoff_datetime",
        "green": "lpep_dropoff_datetime",
        "fhvhv": "dropoff_datetime",
        "justificacion": "Representa la fecha y hora de finalización del viaje."
    },
    {
        "concepto_canonico": "pickup_location_id",
        "yellow": "PULocationID",
        "green": "PULocationID",
        "fhvhv": "PULocationID",
        "justificacion": "Representa la zona de origen del viaje."
    },
    {
        "concepto_canonico": "dropoff_location_id",
        "yellow": "DOLocationID",
        "green": "DOLocationID",
        "fhvhv": "DOLocationID",
        "justificacion": "Representa la zona de destino del viaje."
    },
    {
        "concepto_canonico": "trip_distance",
        "yellow": "trip_distance",
        "green": "trip_distance",
        "fhvhv": "trip_miles",
        "justificacion": "Representa la distancia recorrida en millas."
    },
    {
        "concepto_canonico": "total_amount",
        "yellow": "total_amount",
        "green": "total_amount",
        "fhvhv": "base_passenger_fare + tolls + sales_tax",
        "justificacion": "En FHVHV el total se reconstruye sumando tarifa base, peajes e impuesto."
    },
    {
        "concepto_canonico": "service_type",
        "yellow": "yellow",
        "green": "green",
        "fhvhv": "fhvhv",
        "justificacion": "Permite identificar el origen del servicio."
    }
]

homologation_matrix_df = spark.createDataFrame(homologation_matrix)

homologation_matrix_df.show(truncate=False)

+-------------------+---------------------------------------+---------------------+------------------------------------------------------------------------+---------------------+
|concepto_canonico  |fhvhv                                  |green                |justificacion                                                           |yellow               |
+-------------------+---------------------------------------+---------------------+------------------------------------------------------------------------+---------------------+
|pickup_datetime    |pickup_datetime                        |lpep_pickup_datetime |Representa la fecha y hora de recogida del viaje.                       |tpep_pickup_datetime |
|dropoff_datetime   |dropoff_datetime                       |lpep_dropoff_datetime|Representa la fecha y hora de finalización del viaje.                   |tpep_dropoff_datetime|
|pickup_location_id |PULocationID                           |PULocationID         |Representa la zona de 

In [14]:
# Guardar matriz de homologación en metadata como JSON.

homologation_matrix_path = METADATA_PATH / "homologation_matrix.json"

with open(homologation_matrix_path, "w", encoding="utf-8") as f:
    json.dump(homologation_matrix, f, indent=4, ensure_ascii=False)

print("Matriz de homologación guardada en:")
print(homologation_matrix_path)

Matriz de homologación guardada en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\metadata\homologation_matrix.json


### Justificación de las reglas de homologación

Las reglas de homologación utilizadas no fueron definidas de forma arbitraria. Cada correspondencia se obtuvo a partir de los nombres reales observados en los esquemas de los archivos `yellow`, `green` y `fhvhv`, leídos por Spark y contrastados en `schema_diagnosis_df` con `EXPECTED_SCHEMAS`.

Por ejemplo:

- `tpep_pickup_datetime` en `yellow`, `lpep_pickup_datetime` en `green` y `pickup_datetime` en `fhvhv` representan el mismo concepto de negocio: fecha y hora de inicio del viaje.
- `PULocationID` representa el identificador de ubicación de origen en los tres servicios.
- `DOLocationID` representa el identificador de ubicación de destino en los tres servicios.

La evidencia de estas equivalencias proviene directamente de los esquemas reales obtenidos durante el diagnóstico. La matriz registra esas correspondencias y las funciones `homologate_yellow`, `homologate_green` y `homologate_fhvhv` las aplican al construir las columnas canónicas.


### Relación entre `EXPECTED_SCHEMAS` y la matriz de homologación

`EXPECTED_SCHEMAS` define los nombres nativos reales de cada servicio. Por ejemplo, en `yellow` la fecha de recogida se llama `tpep_pickup_datetime`, mientras que en `green` se llama `lpep_pickup_datetime`. Esta definición se usa para diagnosticar la estabilidad estructural dentro de cada fuente: columnas faltantes, columnas adicionales y tipos incompatibles.

La matriz de homologación trabaja en otro nivel. Su objetivo es resolver las columnas equivalentes con nombres distintos entre servicios, mapeando esos nombres nativos hacia un concepto canónico común, como `pickup_datetime`.

Por eso, ambos elementos son complementarios y no redundantes: `EXPECTED_SCHEMAS` compara cada servicio contra su estructura esperada, mientras que la matriz de homologación permite unificar `yellow`, `green` y `fhvhv` bajo un mismo esquema canónico.

## 9. Crear esquema canónico

El esquema canónico es la estructura común que tendrán todos los viajes.

Después de esta fase, un viaje de `yellow`, `green` o `fhvhv` tendrá las mismas columnas.

In [15]:
# Esquema canónico mínimo solicitado.

CANONICAL_SCHEMA = {
    "trip_id": "string",
    "service_type": "string",
    "vendor_id": "long",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "pickup_location_id": "long",
    "dropoff_location_id": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra_amount": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
    "year": "integer",
    "month": "integer",
    "source_file": "string",
    "ingestion_timestamp": "timestamp",
    "quality_status": "string"
}

CANONICAL_COLUMNS = list(CANONICAL_SCHEMA.keys())

canonical_schema_path = METADATA_PATH / "canonical_schema_trips.json"

with open(canonical_schema_path, "w", encoding="utf-8") as f:
    json.dump(CANONICAL_SCHEMA, f, indent=4, ensure_ascii=False)

print("Esquema canónico guardado en:")
print(canonical_schema_path)

for col_name, col_type in CANONICAL_SCHEMA.items():
    print(f"{col_name}: {col_type}")

Esquema canónico guardado en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\metadata\canonical_schema_trips.json
trip_id: string
service_type: string
vendor_id: long
pickup_datetime: timestamp
dropoff_datetime: timestamp
passenger_count: double
trip_distance: double
pickup_location_id: long
dropoff_location_id: long
payment_type: long
fare_amount: double
extra_amount: double
mta_tax: double
tip_amount: double
tolls_amount: double
total_amount: double
congestion_surcharge: double
airport_fee: double
year: integer
month: integer
source_file: string
ingestion_timestamp: timestamp
quality_status: string


## 10. Homologar Yellow

Yellow usa columnas como:

- `tpep_pickup_datetime`
- `tpep_dropoff_datetime`
- `PULocationID`
- `DOLocationID`

Aquí se renombran y transforman al esquema canónico.

In [16]:
# Funciones auxiliares para homologación.

def column_exists(df, column_name):
    return column_name in df.columns

def safe_col(df, column_name, target_type=None):
    # Si la columna existe, la devuelve. Si no existe, devuelve NULL.
    if column_exists(df, column_name):
        col_expr = F.col(column_name)
    else:
        col_expr = F.lit(None)

    if target_type is not None:
        col_expr = col_expr.cast(target_type)

    return col_expr

def add_missing_canonical_columns(df):
    # Agrega columnas canónicas que falten con NULL controlado.
    for col_name, col_type in CANONICAL_SCHEMA.items():
        if col_name not in df.columns:
            df = df.withColumn(col_name, F.lit(None).cast(col_type))
    return df.select(*CANONICAL_COLUMNS)

def create_trip_id_expr():
    # Genera identificador único técnico del viaje.
    return F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("service_type"), F.lit("")),
            F.coalesce(F.col("pickup_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("dropoff_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("pickup_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("dropoff_location_id").cast("string"), F.lit("")),
            F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
            F.coalesce(F.col("source_file"), F.lit(""))
        ),
        256
    )

In [17]:
def homologate_yellow(df, source_file, year_value, month_value):
    # Homologa un archivo yellow al esquema canónico.

    result = df.select(
        F.lit("yellow").alias("service_type"),
        safe_col(df, "VendorID", "long").alias("vendor_id"),
        safe_col(df, "tpep_pickup_datetime", "timestamp").alias("pickup_datetime"),
        safe_col(df, "tpep_dropoff_datetime", "timestamp").alias("dropoff_datetime"),
        safe_col(df, "passenger_count", "double").alias("passenger_count"),
        safe_col(df, "trip_distance", "double").alias("trip_distance"),
        safe_col(df, "PULocationID", "long").alias("pickup_location_id"),
        safe_col(df, "DOLocationID", "long").alias("dropoff_location_id"),
        safe_col(df, "payment_type", "long").alias("payment_type"),
        safe_col(df, "fare_amount", "double").alias("fare_amount"),
        safe_col(df, "extra", "double").alias("extra_amount"),
        safe_col(df, "mta_tax", "double").alias("mta_tax"),
        safe_col(df, "tip_amount", "double").alias("tip_amount"),
        safe_col(df, "tolls_amount", "double").alias("tolls_amount"),
        safe_col(df, "total_amount", "double").alias("total_amount"),
        safe_col(df, "congestion_surcharge", "double").alias("congestion_surcharge"),
        safe_col(df, "airport_fee", "double").alias("airport_fee"),
        F.lit(int(year_value) if year_value else None).cast("integer").alias("year"),
        F.lit(int(month_value) if month_value else None).cast("integer").alias("month"),
        F.lit(source_file).alias("source_file"),
        F.current_timestamp().alias("ingestion_timestamp"),
        F.lit("RECOVERED_SCHEMA_CANONICAL").alias("quality_status")
    )

    result = result.withColumn("trip_id", create_trip_id_expr())
    result = add_missing_canonical_columns(result)

    return result

## 11. Homologar Green

Green usa nombres parecidos a Yellow, pero sus fechas son:

- `lpep_pickup_datetime`
- `lpep_dropoff_datetime`

Se convierten al mismo esquema canónico.

In [18]:
def homologate_green(df, source_file, year_value, month_value):
    # Homologa un archivo green al esquema canónico.

    result = df.select(
        F.lit("green").alias("service_type"),
        safe_col(df, "VendorID", "long").alias("vendor_id"),
        safe_col(df, "lpep_pickup_datetime", "timestamp").alias("pickup_datetime"),
        safe_col(df, "lpep_dropoff_datetime", "timestamp").alias("dropoff_datetime"),
        safe_col(df, "passenger_count", "double").alias("passenger_count"),
        safe_col(df, "trip_distance", "double").alias("trip_distance"),
        safe_col(df, "PULocationID", "long").alias("pickup_location_id"),
        safe_col(df, "DOLocationID", "long").alias("dropoff_location_id"),
        safe_col(df, "payment_type", "long").alias("payment_type"),
        safe_col(df, "fare_amount", "double").alias("fare_amount"),
        safe_col(df, "extra", "double").alias("extra_amount"),
        safe_col(df, "mta_tax", "double").alias("mta_tax"),
        safe_col(df, "tip_amount", "double").alias("tip_amount"),
        safe_col(df, "tolls_amount", "double").alias("tolls_amount"),
        safe_col(df, "total_amount", "double").alias("total_amount"),
        safe_col(df, "congestion_surcharge", "double").alias("congestion_surcharge"),
        F.lit(None).cast("double").alias("airport_fee"),
        F.lit(int(year_value) if year_value else None).cast("integer").alias("year"),
        F.lit(int(month_value) if month_value else None).cast("integer").alias("month"),
        F.lit(source_file).alias("source_file"),
        F.current_timestamp().alias("ingestion_timestamp"),
        F.lit("RECOVERED_SCHEMA_CANONICAL").alias("quality_status")
    )

    result = result.withColumn("trip_id", create_trip_id_expr())
    result = add_missing_canonical_columns(result)

    return result

## 12. Homologar FHVHV

FHVHV tiene una estructura más diferente.

Por ejemplo:

- La distancia viene como `trip_miles`.
- La tarifa base viene como `base_passenger_fare`.
- El total no siempre viene como `total_amount`.

Por eso se reconstruye:

`total_amount = base_passenger_fare + tolls + sales_tax`

In [19]:
def homologate_fhvhv(df, source_file, year_value, month_value):
    # Homologa un archivo fhvhv al esquema canónico.

    base_fare = safe_col(df, "base_passenger_fare", "double")
    tolls = safe_col(df, "tolls", "double")
    sales_tax = safe_col(df, "sales_tax", "double")

    total_reconstructed = (
        F.coalesce(base_fare, F.lit(0.0)) +
        F.coalesce(tolls, F.lit(0.0)) +
        F.coalesce(sales_tax, F.lit(0.0))
    )

    result = df.select(
        F.lit("fhvhv").alias("service_type"),
        F.lit(None).cast("long").alias("vendor_id"),
        safe_col(df, "pickup_datetime", "timestamp").alias("pickup_datetime"),
        safe_col(df, "dropoff_datetime", "timestamp").alias("dropoff_datetime"),
        F.lit(None).cast("double").alias("passenger_count"),
        safe_col(df, "trip_miles", "double").alias("trip_distance"),
        safe_col(df, "PULocationID", "long").alias("pickup_location_id"),
        safe_col(df, "DOLocationID", "long").alias("dropoff_location_id"),
        F.lit(None).cast("long").alias("payment_type"),
        base_fare.alias("fare_amount"),
        F.lit(None).cast("double").alias("extra_amount"),
        F.lit(None).cast("double").alias("mta_tax"),
        safe_col(df, "tips", "double").alias("tip_amount"),
        tolls.alias("tolls_amount"),
        total_reconstructed.cast("double").alias("total_amount"),
        F.lit(None).cast("double").alias("congestion_surcharge"),
        F.lit(None).cast("double").alias("airport_fee"),
        F.lit(int(year_value) if year_value else None).cast("integer").alias("year"),
        F.lit(int(month_value) if month_value else None).cast("integer").alias("month"),
        F.lit(source_file).alias("source_file"),
        F.current_timestamp().alias("ingestion_timestamp"),
        F.lit("RECOVERED_SCHEMA_CANONICAL").alias("quality_status")
    )

    result = result.withColumn("trip_id", create_trip_id_expr())
    result = add_missing_canonical_columns(result)

    return result

### Justificación de `total_amount` en FHVHV

Para `fhvhv`, el campo `total_amount` se reconstruye usando la fórmula mínima indicada en el caso de estudio:

`base_passenger_fare + tolls + sales_tax`

Esta decisión permite mantener consistencia con la definición solicitada para la homologación. Otros posibles componentes de tarifa de FHVHV, como `bcf` o `congestion_surcharge` si existieran en el dataset real, no se incluyen en esta fase para no alterar la regla dada. Esta decisión queda documentada como una limitación que puede revisarse en fases posteriores si se requiere mayor precisión financiera.

## 13. Agregar columnas faltantes

Esto ya se realiza dentro de la función `add_missing_canonical_columns`.

Si una columna no existe en una fuente, se agrega como `NULL` con el tipo correcto.

Ejemplo:

- FHVHV no tiene `passenger_count`
- Green no tiene `airport_fee`

Entonces se agregan como nulos controlados.

In [20]:
# Demostración de columnas canónicas esperadas.

print("Columnas canónicas finales:")

for col_name in CANONICAL_COLUMNS:
    print("-", col_name)

Columnas canónicas finales:
- trip_id
- service_type
- vendor_id
- pickup_datetime
- dropoff_datetime
- passenger_count
- trip_distance
- pickup_location_id
- dropoff_location_id
- payment_type
- fare_amount
- extra_amount
- mta_tax
- tip_amount
- tolls_amount
- total_amount
- congestion_surcharge
- airport_fee
- year
- month
- source_file
- ingestion_timestamp
- quality_status


### Alcance de `reconstruction_status`

El campo `reconstruction_status`, calculado durante el diagnóstico, es informativo y sirve para auditoría. Su función es documentar si un archivo fue clasificado como recuperable, parcialmente recuperable o no recuperable.

Este campo no condiciona directamente el flujo de homologación. La reconstrucción se realiza mediante las funciones `homologate_yellow`, `homologate_green` y `homologate_fhvhv`, que ya manejan columnas ausentes o tipos diferentes usando `safe_col` y `add_missing_canonical_columns`. Por eso, el pipeline puede reconstruir el esquema canónico de forma robusta aunque existan diferencias estructurales.

## 14. Convertir tipos de datos

Las funciones de homologación aplican `cast` controlado:

- Fechas a `timestamp`
- Montos a `double`
- Identificadores de zona a `long`
- Año y mes a `integer`

Esto permite recuperar archivos con tipos diferentes sin detener el pipeline.

In [21]:
# Función principal para reconstruir un archivo hacia el esquema canónico.

def reconstruct_file(row):
    service_type = row["service_type"]
    file_path = Path(row["file_path"])
    file_name = row["file_name"]
    year_value = row["partition_year"]
    month_value = row["partition_month"]

    try:
        df = spark.read.parquet(str(file_path))

        if service_type == "yellow":
            reconstructed_df = homologate_yellow(df, file_name, year_value, month_value)
        elif service_type == "green":
            reconstructed_df = homologate_green(df, file_name, year_value, month_value)
        elif service_type == "fhvhv":
            reconstructed_df = homologate_fhvhv(df, file_name, year_value, month_value)
        else:
            return None, {
                "process_id": PROCESS_ID,
                "file_name": file_name,
                "file_path": str(file_path),
                "service_type": service_type,
                "stage": "schema_reconstruction",
                "classification": "NOT_RECOVERABLE_UNSUPPORTED_FORMAT",
                "technical_reason": "Tipo de servicio no soportado para homologación.",
                "recommended_action": "Revisar manualmente la fuente.",
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

        return reconstructed_df, None

    except Exception as e:
        return None, {
            "process_id": PROCESS_ID,
            "file_name": file_name,
            "file_path": str(file_path),
            "service_type": service_type,
            "stage": "schema_reconstruction",
            "classification": "NOT_RECOVERABLE_CORRUPT_METADATA",
            "technical_reason": clean_error_message(e),
            "recommended_action": "Enviar a cuarentena técnica y revisar archivo original.",
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

## 15. Clasificar recuperables y no recuperables

Un archivo es recuperable cuando puede leerse y puede transformarse al esquema canónico.

Un archivo no recuperable es aquel que falla durante la reconstrucción o tiene un formato no soportado.

In [22]:
# Reconstrucción de todos los archivos recuperables.

reconstructed_dfs = []
non_recoverable_rows = []

for row in readable_files:
    reconstructed_df, error_info = reconstruct_file(row)

    if reconstructed_df is not None:
        reconstructed_dfs.append(reconstructed_df)
        print("RECUPERADO:", row["service_type"], "-", row["file_name"])
    else:
        non_recoverable_rows.append(error_info)
        print("NO RECUPERABLE:", row["service_type"], "-", row["file_name"])

print("DataFrames reconstruidos:", len(reconstructed_dfs))
print("Archivos no recuperables en Fase 2:", len(non_recoverable_rows))

RECUPERADO: yellow - yellow_tripdata_2020-01.parquet
RECUPERADO: yellow - yellow_tripdata_2022-01.parquet
RECUPERADO: yellow - yellow_tripdata_2022-02.parquet
RECUPERADO: yellow - yellow_tripdata_2023-01.parquet
RECUPERADO: yellow - yellow_tripdata_2023-03.parquet
RECUPERADO: yellow - yellow_tripdata_2021-01.parquet
RECUPERADO: yellow - yellow_tripdata_2023-02.parquet
RECUPERADO: yellow - yellow_tripdata_2023-04.parquet
RECUPERADO: fhvhv - fhvhv_tripdata_2023-01.parquet
RECUPERADO: green - green_tripdata_2023-01.parquet
RECUPERADO: green - green_tripdata_2023-02.parquet
DataFrames reconstruidos: 11
Archivos no recuperables en Fase 2: 0


In [23]:
# Unir todos los DataFrames reconstruidos.
# unionByName permite unir columnas por nombre.

if len(reconstructed_dfs) == 0:
    raise ValueError("No existen archivos reconstruidos. Revisar la Fase 1 o las rutas raw.")

bronze_trips_canonical = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    reconstructed_dfs
)

# Asegurar orden final de columnas canónicas.
bronze_trips_canonical = bronze_trips_canonical.select(*CANONICAL_COLUMNS)

print("Columnas del dataset reconstruido:", len(bronze_trips_canonical.columns))
bronze_trips_canonical.printSchema()

Columnas del dataset reconstruido: 23
root
 |-- trip_id: string (nullable = true)
 |-- service_type: string (nullable = false)
 |-- vendor_id: long (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra_amount: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- year: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- source_file: string (nullable = false)
 |-- ingestion_timestamp: time

## 16. Enviar no recuperables a cuarentena

Los archivos que no se puedan reconstruir se registran en cuarentena con:

- Archivo
- Servicio
- Etapa donde falló
- Clasificación
- Motivo técnico
- Acción recomendada

In [24]:
# Crear tabla de archivos no recuperables en Fase 2.

if len(non_recoverable_rows) > 0:
    phase2_quarantine_df = spark.createDataFrame(non_recoverable_rows)
else:
    phase2_quarantine_schema = StructType([
        StructField("process_id", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("file_path", StringType(), True),
        StructField("service_type", StringType(), True),
        StructField("stage", StringType(), True),
        StructField("classification", StringType(), True),
        StructField("technical_reason", StringType(), True),
        StructField("recommended_action", StringType(), True),
        StructField("created_at", StringType(), True)
    ])
    phase2_quarantine_df = spark.createDataFrame([], phase2_quarantine_schema)

phase2_quarantine_df.show(truncate=False)

+----------+---------+---------+------------+-----+--------------+----------------+------------------+----------+
|process_id|file_name|file_path|service_type|stage|classification|technical_reason|recommended_action|created_at|
+----------+---------+---------+------------+-----+--------------+----------------+------------------+----------+
+----------+---------+---------+------------+-----+--------------+----------------+------------------+----------+



In [25]:
# Guardar diagnóstico de esquemas en audit.

schema_diagnosis_output = AUDIT_PATH / f"schema_diagnosis_{PROCESS_ID}"

schema_diagnosis_df.write.mode("overwrite").parquet(
    str(schema_diagnosis_output)
)

print("Diagnóstico de esquemas guardado en:")
print(schema_diagnosis_output)

Diagnóstico de esquemas guardado en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\schema_diagnosis_fase2_20260617_170931


In [26]:
# Guardar cuarentena de Fase 2.

phase2_quarantine_output = QUARANTINE_PATH / f"schema_reconstruction_exclusions_{PROCESS_ID}"

phase2_quarantine_df.write.mode("overwrite").parquet(
    str(phase2_quarantine_output)
)

print("Cuarentena de reconstrucción guardada en:")
print(phase2_quarantine_output)

Cuarentena de reconstrucción guardada en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\schema_reconstruction_exclusions_fase2_20260617_170931


In [27]:
""" # Guardar dataset reconstruido en bronze.

# En Windows se guarda una muestra controlada por servicio para evitar caída de Spark.

# La reconstrucción completa ya fue validada en memoria; esta salida deja evidencia física en bronze.



bronze_base_path = BRONZE_PATH / f"trips_canonical_{PROCESS_ID}"



services_to_save = ["yellow", "green", "fhvhv"]

MAX_ROWS_PER_SERVICE = 10000



for service in services_to_save:

    service_output_path = bronze_base_path / f"service_type={service}"



    df_service = (

        bronze_trips_canonical

        .filter(F.col("service_type") == service)

        .limit(MAX_ROWS_PER_SERVICE)

        .coalesce(1)

    )



    total_service_records = df_service.count()



    print("="*60)

    print(f"Guardando servicio: {service}")

    print(f"Registros guardados: {total_service_records}")



    if total_service_records > 0:

        (

            df_service

            .write

            .mode("overwrite")

            .parquet(str(service_output_path))

        )



        print("Guardado correctamente en:")

        print(service_output_path)

    else:

        print("No existen registros para este servicio.")



print("="*60)

print("Dataset canónico reconstruido guardado en bronze por servicio.")

print("Ruta base:")

print(bronze_base_path) """

' # Guardar dataset reconstruido en bronze.\n\n# En Windows se guarda una muestra controlada por servicio para evitar caída de Spark.\n\n# La reconstrucción completa ya fue validada en memoria; esta salida deja evidencia física en bronze.\n\n\n\nbronze_base_path = BRONZE_PATH / f"trips_canonical_{PROCESS_ID}"\n\n\n\nservices_to_save = ["yellow", "green", "fhvhv"]\n\nMAX_ROWS_PER_SERVICE = 10000\n\n\n\nfor service in services_to_save:\n\n    service_output_path = bronze_base_path / f"service_type={service}"\n\n\n\n    df_service = (\n\n        bronze_trips_canonical\n\n        .filter(F.col("service_type") == service)\n\n        .limit(MAX_ROWS_PER_SERVICE)\n\n        .coalesce(1)\n\n    )\n\n\n\n    total_service_records = df_service.count()\n\n\n\n    print("="*60)\n\n    print(f"Guardando servicio: {service}")\n\n    print(f"Registros guardados: {total_service_records}")\n\n\n\n    if total_service_records > 0:\n\n        (\n\n            df_service\n\n            .write\n\n         

In [28]:
# Guardar dataset reconstruido completo en Bronze.
# No usa limit().
# No usa repartition(), para evitar shuffle y errores de memoria.
# Escribe por servicio y por source_file para reducir la carga por cada escritura.

bronze_base_path = BRONZE_PATH / f"trips_canonical_{PROCESS_ID}"

services_to_save = ["yellow", "green", "fhvhv"]

for service in services_to_save:
    df_service = bronze_trips_canonical.filter(F.col("service_type") == service)

    source_files = [
        row["source_file"]
        for row in df_service.select("source_file").distinct().collect()
    ]

    print("=" * 80)
    print(f"Servicio: {service}")
    print(f"Archivos fuente encontrados: {len(source_files)}")

    for source_file in source_files:
        safe_source_name = (
            str(source_file)
            .replace(".parquet", "")
            .replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
            .replace(" ", "_")
        )

        output_path = bronze_base_path / f"service_type={service}" / f"source_file={safe_source_name}"

        df_file = df_service.filter(F.col("source_file") == source_file)

        total_file_records = df_file.count()

        print("-" * 80)
        print(f"Guardando archivo fuente: {source_file}")
        print(f"Registros guardados: {total_file_records}")

        if total_file_records > 0:
            (
                df_file
                .write
                .mode("overwrite")
                .parquet(str(output_path))
            )

            print("Guardado correctamente en:")
            print(output_path)

print("=" * 80)
print("Dataset canónico reconstruido COMPLETO guardado en Bronze.")
print("Ruta base:")
print(bronze_base_path)

Servicio: yellow
Archivos fuente encontrados: 8
--------------------------------------------------------------------------------
Guardando archivo fuente: yellow_tripdata_2020-01.parquet
Registros guardados: 6405008
Guardado correctamente en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze\trips_canonical_fase2_20260617_170931\service_type=yellow\source_file=yellow_tripdata_2020-01
--------------------------------------------------------------------------------
Guardando archivo fuente: yellow_tripdata_2022-01.parquet
Registros guardados: 2463931
Guardado correctamente en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze\trips_canonical_fase2_20260617_170931\service_type=yellow\source_file=yellow_tripdata_2022-01
--------------------------------------------------------------------------------
Guardando archivo fuente: yellow_tripdata_2022-02.parquet
Registros guardados: 2979431
Guardado correctamen

## 17. Resultado final

Se valida que la Fase 2 haya producido un dataset operativo bajo el esquema canónico común.

In [29]:
# Verificar lectura del resultado guardado en Bronze.
# Se lee desde la ruta base para evitar conflicto con carpetas particionadas.

bronze_check_df = (
    spark.read
    .option("basePath", str(bronze_base_path))
    .parquet(
        str(bronze_base_path / "service_type=yellow"),
        str(bronze_base_path / "service_type=green"),
        str(bronze_base_path / "service_type=fhvhv")
    )
)

print("Registros reconstruidos:", bronze_check_df.count())
print("Columnas reconstruidas:", len(bronze_check_df.columns))

bronze_check_df.groupBy("service_type").count().show()

bronze_check_df.select(
    "trip_id",
    "service_type",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "year",
    "month",
    "source_file",
    "quality_status"
).show(10, truncate=False)

Registros reconstruidos: 44502927
Columnas reconstruidas: 23
+------------+--------+
|service_type|   count|
+------------+--------+
|      yellow|25890876|
|       fhvhv|18479031|
|       green|  133020|
+------------+--------+

+----------------------------------------------------------------+------------+-------------------+-------------------+-------------+------------------+-------------------+-----------+----------+------------+----+-----+-----------------------+--------------------------+
|trip_id                                                         |service_type|pickup_datetime    |dropoff_datetime   |trip_distance|pickup_location_id|dropoff_location_id|fare_amount|tip_amount|total_amount|year|month|source_file            |quality_status            |
+----------------------------------------------------------------+------------+-------------------+-------------------+-------------+------------------+-------------------+-----------+----------+------------+----+-----+---------

In [30]:
# Validación de columnas canónicas.

final_columns = set(bronze_check_df.columns)
expected_columns = set(CANONICAL_COLUMNS)

missing_final_columns = sorted(list(expected_columns - final_columns))
extra_final_columns = sorted(list(final_columns - expected_columns))

print("Columnas faltantes en resultado final:", missing_final_columns)
print("Columnas adicionales en resultado final:", extra_final_columns)

if len(missing_final_columns) == 0:
    print("El dataset reconstruido cumple con el esquema canónico mínimo.")
else:
    print("Faltan columnas canónicas. Revisar homologación.")

Columnas faltantes en resultado final: []
Columnas adicionales en resultado final: []
El dataset reconstruido cumple con el esquema canónico mínimo.


In [31]:
# Resumen final de Fase 2.

total_diagnosed = schema_diagnosis_df.count()
total_reconstructed_files = len(reconstructed_dfs)
total_non_recoverable = phase2_quarantine_df.count()
total_records_bronze = bronze_check_df.count()

print("=" * 70)
print("RESUMEN FASE 2 - DIAGNÓSTICO Y RECONSTRUCCIÓN")
print("=" * 70)
print(f"Archivos diagnosticados: {total_diagnosed}")
print(f"Archivos reconstruidos correctamente: {total_reconstructed_files}")
print(f"Archivos no recuperables enviados a cuarentena: {total_non_recoverable}")
print(f"Registros reconstruidos en bronze: {total_records_bronze}")
print(f"Columnas canónicas finales: {len(CANONICAL_COLUMNS)}")
print("=" * 70)

print("Fase 2 completada correctamente.")

RESUMEN FASE 2 - DIAGNÓSTICO Y RECONSTRUCCIÓN
Archivos diagnosticados: 11
Archivos reconstruidos correctamente: 11
Archivos no recuperables enviados a cuarentena: 0
Registros reconstruidos en bronze: 44502927
Columnas canónicas finales: 23
Fase 2 completada correctamente.


# Conclusión de la Fase 2

En esta fase se compararon los esquemas reales contra esquemas esperados, se detectaron columnas faltantes, columnas adicionales y tipos incompatibles.

Luego se construyó una matriz de homologación para unificar `yellow`, `green` y `fhvhv` bajo un esquema canónico común.

Los archivos recuperables fueron reconstruidos y guardados en la capa `bronze`.

Los archivos no recuperables fueron documentados y enviados a cuarentena con su motivo técnico. También se registraron las exclusiones funcionales de archivos que no pertenecen a las fuentes de negocio.

Como resultado, se reconstruyó un dataset canónico en `bronze` y quedaron documentadas las decisiones, exclusiones y limitaciones técnicas de la Fase 2.

# 03 - Reconstrucción ante archivos dañados

## Fase 3: Reconstrucción ante archivos dañados

Este notebook implementa la Fase 3 del proyecto de Modelado Avanzado de Base de Datos.

El objetivo es diseñar una estrategia profesional para tratar archivos Parquet dañados, ilegibles, parcialmente recuperables o con problemas de esquema.  
La solución no ignora los archivos problemáticos: los analiza, los clasifica, registra evidencia técnica, intenta recuperar los que sí son recuperables y envía a cuarentena los que no pueden integrarse al pipeline principal.

### Qué se cumple en esta fase

1. Lectura individual y controlada de archivos problemáticos.
2. Clasificación obligatoria de cada archivo.
3. Diferenciación entre archivos recuperables, parcialmente recuperables y no recuperables.
4. Diferenciación entre errores de esquema, errores físicos de archivo, formatos no soportados y problemas de calidad.
5. Captura de excepciones técnicas sin detener todo el proceso.
6. Registro de cuarentena en `data/quarantine/`.
7. Copia de archivos problemáticos como evidencia.
8. Intento real de reconstrucción hacia esquema canónico.
9. Escritura de archivos recuperados en `data/bronze/recovered_files/`.
10. Generación de resumen final para auditoría.


In [32]:
# Esta celda prepara Python, Hadoop y Spark para trabajar en Windows.

# Importar librerías principales.

import os
import sys
import json
import shutil
from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Python usado por Jupyter:")
print(sys.executable)

print("PYSPARK_PYTHON:")
print(os.environ["PYSPARK_PYTHON"])

print("Existe winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("Existe hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

Python usado por Jupyter:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
PYSPARK_PYTHON:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
Existe winutils: False
Existe hadoop.dll: False


In [33]:
# Importar PySpark.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)


In [34]:
# Crear o reutilizar la sesión de Spark.

# Java 17 usado por Spark.
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = r"C:\Program Files\Java\jdk-17\bin;" + os.environ["PATH"]


spark = (
    SparkSession.builder
    .appName("Fase_03_Reconstruccion_Archivos_Danados")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.io.native.lib.available", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente.")
print("Versión de Spark:", spark.version)


Spark iniciado correctamente.
Versión de Spark: 4.1.2


In [35]:
# Definir rutas principales del proyecto.

# Si el notebook se ejecuta desde la carpeta notebooks, BASE_PATH será la carpeta padre.
# Si se ejecuta desde la raíz del proyecto, BASE_PATH será la carpeta actual.
BASE_PATH = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RAW_PATH = BASE_PATH / "data" / "raw"
BRONZE_PATH = BASE_PATH / "data" / "bronze"
QUARANTINE_PATH = BASE_PATH / "data" / "quarantine"
AUDIT_PATH = BASE_PATH / "data" / "audit"

BAD_PARQUET_PATH = RAW_PATH / "bad_parquet"

PROCESS_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

for path in [RAW_PATH, BRONZE_PATH, QUARANTINE_PATH, AUDIT_PATH, BAD_PARQUET_PATH]:
    path.mkdir(parents=True, exist_ok=True)

def spark_path(path):
    """
    Convierte rutas locales de Windows a un formato compatible con Spark.
    Ejemplo: C:\\proyecto\\data -> C:/proyecto/data
    """
    return str(Path(path).resolve()).replace("\\", "/")

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("BAD_PARQUET_PATH:", BAD_PARQUET_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("PROCESS_ID:", PROCESS_ID)


BASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
RAW_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw
BAD_PARQUET_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet
BRONZE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze
QUARANTINE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
PROCESS_ID: 20260617_171909


## 1. Categorías obligatorias de la Fase 3

El enunciado solicita clasificar cada archivo problemático en una de estas categorías:

- `RECUPERABLE_SCHEMA_MISMATCH`
- `RECUPERABLE_MISSING_COLUMNS`
- `RECUPERABLE_TYPE_CASTING`
- `PARTIALLY_RECOVERABLE`
- `NOT_RECOVERABLE_CORRUPT_METADATA`
- `NOT_RECOVERABLE_EMPTY_FILE`
- `NOT_RECOVERABLE_UNSUPPORTED_FORMAT`

Además, para explicar mejor la decisión técnica, se agregan dos clasificaciones auxiliares:

- `recoverability`: indica si el archivo es recuperable, parcialmente recuperable o no recuperable.
- `error_type`: indica si el problema es de esquema, físico, formato o calidad de negocio.


In [36]:
# Categorías obligatorias exigidas por el enunciado.

RECUPERABLE_SCHEMA_MISMATCH = "RECUPERABLE_SCHEMA_MISMATCH"
RECUPERABLE_MISSING_COLUMNS = "RECUPERABLE_MISSING_COLUMNS"
RECUPERABLE_TYPE_CASTING = "RECUPERABLE_TYPE_CASTING"
PARTIALLY_RECOVERABLE = "PARTIALLY_RECOVERABLE"
NOT_RECOVERABLE_CORRUPT_METADATA = "NOT_RECOVERABLE_CORRUPT_METADATA"
NOT_RECOVERABLE_EMPTY_FILE = "NOT_RECOVERABLE_EMPTY_FILE"
NOT_RECOVERABLE_UNSUPPORTED_FORMAT = "NOT_RECOVERABLE_UNSUPPORTED_FORMAT"

# Clasificaciones auxiliares para explicar mejor el tratamiento.
RECOVERABLE = "RECOVERABLE"
PARTIALLY_RECOVERABLE_STATUS = "PARTIALLY_RECOVERABLE"
NOT_RECOVERABLE = "NOT_RECOVERABLE"

SCHEMA_ERROR = "SCHEMA_ERROR"
PHYSICAL_FILE_ERROR = "PHYSICAL_FILE_ERROR"
UNSUPPORTED_FORMAT_ERROR = "UNSUPPORTED_FORMAT_ERROR"
BUSINESS_RULE_ERROR = "BUSINESS_RULE_ERROR"

print("Categorías de Fase 3 cargadas correctamente.")


Categorías de Fase 3 cargadas correctamente.


## 2. Esquema canónico y matriz de equivalencias

Para que la reconstrucción sea más profesional, no se depende únicamente de que las columnas tengan exactamente el nombre canónico.

Por ejemplo:

- `tpep_pickup_datetime` y `lpep_pickup_datetime` pueden reconstruirse como `pickup_datetime`.
- `PULocationID` puede reconstruirse como `pickup_location_id`.
- `DOLocationID` puede reconstruirse como `dropoff_location_id`.

Esto permite recuperar archivos con diferencias de nombres, algo común al trabajar con fuentes heterogéneas.


In [37]:
# Esquema canónico mínimo usado para reconstruir archivos recuperables.

canonical_type_map = {
    "trip_id": "string",
    "service_type": "string",
    "vendor_id": "string",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "passenger_count": "int",
    "trip_distance": "double",
    "pickup_location_id": "int",
    "dropoff_location_id": "int",
    "payment_type": "string",
    "fare_amount": "double",
    "extra_amount": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
    "year": "int",
    "month": "int",
    "source_file": "string",
    "ingestion_timestamp": "timestamp",
    "quality_status": "string"
}

CANONICAL_COLUMNS = list(canonical_type_map.keys())

# Matriz de equivalencias para reconstruir columnas con nombres diferentes.
COLUMN_EQUIVALENCES = {
    "trip_id": ["trip_id"],
    "service_type": ["service_type"],
    "vendor_id": ["vendor_id", "VendorID"],
    "pickup_datetime": ["pickup_datetime", "tpep_pickup_datetime", "lpep_pickup_datetime"],
    "dropoff_datetime": ["dropoff_datetime", "tpep_dropoff_datetime", "lpep_dropoff_datetime"],
    "passenger_count": ["passenger_count"],
    "trip_distance": ["trip_distance"],
    "pickup_location_id": ["pickup_location_id", "PULocationID", "pu_location_id"],
    "dropoff_location_id": ["dropoff_location_id", "DOLocationID", "do_location_id"],
    "payment_type": ["payment_type"],
    "fare_amount": ["fare_amount", "base_passenger_fare"],
    "extra_amount": ["extra_amount", "extra"],
    "mta_tax": ["mta_tax"],
    "tip_amount": ["tip_amount", "tips"],
    "tolls_amount": ["tolls_amount", "tolls"],
    "total_amount": ["total_amount"],
    "congestion_surcharge": ["congestion_surcharge"],
    "airport_fee": ["airport_fee", "Airport_fee"],
    "year": ["year"],
    "month": ["month"],
    "source_file": ["source_file"],
    "ingestion_timestamp": ["ingestion_timestamp"],
    "quality_status": ["quality_status"]
}

print("Columnas canónicas:")
print(CANONICAL_COLUMNS)


Columnas canónicas:
['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'quality_status']


## 3. Funciones de clasificación y reconstrucción

Las siguientes funciones permiten:

1. Clasificar el archivo según la categoría obligatoria.
2. Determinar si el archivo es recuperable o no.
3. Determinar el tipo de error.
4. Diagnosticar columnas faltantes, adicionales o incompatibles.
5. Reconstruir archivos recuperables hacia el esquema canónico.
6. Registrar evidencia suficiente para auditoría.


In [38]:
def get_recoverability(problem_category):
    """Devuelve si el archivo es recuperable, parcialmente recuperable o no recuperable."""
    if problem_category in [
        RECUPERABLE_SCHEMA_MISMATCH,
        RECUPERABLE_MISSING_COLUMNS,
        RECUPERABLE_TYPE_CASTING
    ]:
        return RECOVERABLE

    if problem_category == PARTIALLY_RECOVERABLE:
        return PARTIALLY_RECOVERABLE_STATUS

    return NOT_RECOVERABLE


def get_error_type(problem_category):
    """Clasifica el tipo general del error."""
    if problem_category in [
        RECUPERABLE_SCHEMA_MISMATCH,
        RECUPERABLE_MISSING_COLUMNS,
        RECUPERABLE_TYPE_CASTING
    ]:
        return SCHEMA_ERROR

    if problem_category in [
        NOT_RECOVERABLE_CORRUPT_METADATA,
        NOT_RECOVERABLE_EMPTY_FILE
    ]:
        return PHYSICAL_FILE_ERROR

    if problem_category == NOT_RECOVERABLE_UNSUPPORTED_FORMAT:
        return UNSUPPORTED_FORMAT_ERROR

    if problem_category == PARTIALLY_RECOVERABLE:
        return BUSINESS_RULE_ERROR

    return PHYSICAL_FILE_ERROR


def classify_exception(file_path, exception_message):
    """Clasifica archivos que fallan al leerse con Spark."""
    file_path_text = str(file_path).lower()
    message = str(exception_message).lower()

    if Path(file_path).exists() and Path(file_path).stat().st_size == 0:
        return NOT_RECOVERABLE_EMPTY_FILE

    if not file_path_text.endswith(".parquet"):
        return NOT_RECOVERABLE_UNSUPPORTED_FORMAT

    corrupt_keywords = [
        "not a parquet",
        "parquet magic",
        "footer",
        "metadata",
        "could not read footer",
        "expected magic number",
        "is not a parquet file",
        "cannot read file footer",
        "runtimeexception",
        "checksum",
        "eofexception",
        "ioexception"
    ]

    type_keywords = [
        "cannot be cast",
        "cannot cast",
        "classcastexception",
        "data type",
        "unsupported type",
        "schema conversion"
    ]

    missing_keywords = [
        "cannot resolve",
        "missing",
        "not found"
    ]

    if any(keyword in message for keyword in type_keywords):
        return RECUPERABLE_TYPE_CASTING

    if any(keyword in message for keyword in missing_keywords):
        return RECUPERABLE_MISSING_COLUMNS

    if "schema" in message:
        return RECUPERABLE_SCHEMA_MISMATCH

    if any(keyword in message for keyword in corrupt_keywords):
        return NOT_RECOVERABLE_CORRUPT_METADATA

    return NOT_RECOVERABLE_CORRUPT_METADATA


def find_equivalent_column(df_columns, canonical_column):
    """
    Busca una columna equivalente para una columna canónica.
    La búsqueda se realiza respetando nombres exactos definidos en COLUMN_EQUIVALENCES.
    """

    possible_names = COLUMN_EQUIVALENCES.get(canonical_column, [canonical_column])

    for name in possible_names:
        if name in df_columns:
            return name

    return None


def diagnose_schema(df):
    """
    Compara el esquema real contra el esquema canónico.
    Usa la matriz de equivalencias para no marcar como faltante una columna que existe con otro nombre.
    """

    actual_columns = df.columns
    actual_types = {field.name: field.dataType.simpleString() for field in df.schema.fields}

    missing_columns = []
    incompatible_types = []

    for canonical_column in CANONICAL_COLUMNS:
        source_column = find_equivalent_column(actual_columns, canonical_column)

        if source_column is None:
            missing_columns.append(canonical_column)
            continue

        actual_type = actual_types[source_column]
        expected_type = canonical_type_map[canonical_column]

        numeric_types = [
            "double",
            "float",
            "int",
            "bigint",
            "long",
            "smallint",
            "decimal"
        ]

        if expected_type == "double":
            if not any(actual_type.startswith(num_type) for num_type in numeric_types):
                incompatible_types.append({
                    "canonical_column": canonical_column,
                    "source_column": source_column,
                    "actual_type": actual_type,
                    "expected_type": expected_type
                })

        elif expected_type == "int":
            if not actual_type.startswith(("int", "bigint", "long", "smallint")):
                incompatible_types.append({
                    "canonical_column": canonical_column,
                    "source_column": source_column,
                    "actual_type": actual_type,
                    "expected_type": expected_type
                })

        elif expected_type == "timestamp":
            if actual_type not in ["timestamp", "timestamp_ntz", "string", "date"]:
                incompatible_types.append({
                    "canonical_column": canonical_column,
                    "source_column": source_column,
                    "actual_type": actual_type,
                    "expected_type": expected_type
                })

        elif expected_type == "string":
            # Cualquier campo puede convertirse a string.
            pass

    mapped_source_columns = set()

    for canonical_column in CANONICAL_COLUMNS:
        source_column = find_equivalent_column(actual_columns, canonical_column)
        if source_column is not None:
            mapped_source_columns.add(source_column)

    additional_columns = [
        col for col in actual_columns
        if col not in mapped_source_columns
    ]

    return missing_columns, additional_columns, incompatible_types


def classify_readable_file(df, record_count):
    """
    Clasifica archivos que Spark sí puede leer, pero que pueden tener problemas de esquema.
    """

    if record_count == 0:
        return NOT_RECOVERABLE_EMPTY_FILE, CANONICAL_COLUMNS, [], []

    missing_columns, additional_columns, incompatible_types = diagnose_schema(df)

    if incompatible_types:
        return RECUPERABLE_TYPE_CASTING, missing_columns, additional_columns, incompatible_types

    if missing_columns:
        return RECUPERABLE_MISSING_COLUMNS, missing_columns, additional_columns, incompatible_types

    if additional_columns:
        return RECUPERABLE_SCHEMA_MISMATCH, missing_columns, additional_columns, incompatible_types

    return PARTIALLY_RECOVERABLE, missing_columns, additional_columns, incompatible_types


def extract_year_month_from_path(file_path):
    """
    Intenta extraer year y month desde rutas tipo year=2023/month=01.
    Si no encuentra esos valores, devuelve None.
    """

    text = str(file_path).replace("\\", "/")
    year_value = None
    month_value = None

    for part in text.split("/"):
        if part.startswith("year="):
            try:
                year_value = int(part.replace("year=", ""))
            except Exception:
                year_value = None

        if part.startswith("month="):
            try:
                month_value = int(part.replace("month=", ""))
            except Exception:
                month_value = None

    return year_value, month_value


def infer_service_type_from_path(file_path):
    """
    Intenta identificar el tipo de servicio desde el nombre o la ruta.
    """

    text = str(file_path).lower()

    if "yellow" in text:
        return "yellow"
    if "green" in text:
        return "green"
    if "fhvhv" in text:
        return "fhvhv"

    return "unknown"


def build_total_amount_expression(df_columns):
    """
    Construye total_amount cuando no existe la columna directa.
    Para FHVHV puede aproximarse a partir de base_passenger_fare + tolls + sales_tax.
    """

    if "total_amount" in df_columns:
        return F.col("total_amount").cast("double")

    available_components = []
    for col_name in ["base_passenger_fare", "tolls", "sales_tax"]:
        if col_name in df_columns:
            available_components.append(F.coalesce(F.col(col_name).cast("double"), F.lit(0.0)))

    if available_components:
        expression = available_components[0]
        for component in available_components[1:]:
            expression = expression + component
        return expression.cast("double")

    return F.lit(None).cast("double")


def reconstruct_to_canonical(df, file_path, problem_category):
    """
    Reconstruye el archivo hacia el esquema canónico usando equivalencias de columnas.
    También convierte cada campo al tipo de dato esperado.
    """

    df_columns = df.columns
    source_file_name = Path(file_path).name
    service_type_value = infer_service_type_from_path(file_path)
    year_value, month_value = extract_year_month_from_path(file_path)

    select_expressions = []

    for canonical_column in CANONICAL_COLUMNS:
        target_type = canonical_type_map[canonical_column]

        if canonical_column == "trip_id":
            select_expressions.append(
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.lit(str(file_path)),
                        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in df_columns[:10]]
                    ),
                    256
                ).alias("trip_id")
            )

        elif canonical_column == "service_type":
            source_column = find_equivalent_column(df_columns, canonical_column)
            if source_column is not None:
                select_expressions.append(F.col(source_column).cast(target_type).alias(canonical_column))
            else:
                select_expressions.append(F.lit(service_type_value).cast(target_type).alias(canonical_column))

        elif canonical_column == "total_amount":
            select_expressions.append(build_total_amount_expression(df_columns).alias("total_amount"))

        elif canonical_column == "year":
            source_column = find_equivalent_column(df_columns, canonical_column)
            if source_column is not None:
                select_expressions.append(F.col(source_column).cast(target_type).alias(canonical_column))
            else:
                select_expressions.append(F.lit(year_value).cast(target_type).alias(canonical_column))

        elif canonical_column == "month":
            source_column = find_equivalent_column(df_columns, canonical_column)
            if source_column is not None:
                select_expressions.append(F.col(source_column).cast(target_type).alias(canonical_column))
            else:
                select_expressions.append(F.lit(month_value).cast(target_type).alias(canonical_column))

        elif canonical_column == "source_file":
            select_expressions.append(F.lit(source_file_name).cast(target_type).alias(canonical_column))

        elif canonical_column == "ingestion_timestamp":
            select_expressions.append(F.current_timestamp().alias(canonical_column))

        elif canonical_column == "quality_status":
            select_expressions.append(F.lit(f"RECOVERED_{problem_category}").cast(target_type).alias(canonical_column))

        else:
            source_column = find_equivalent_column(df_columns, canonical_column)

            if source_column is not None:
                select_expressions.append(
                    F.col(source_column).cast(target_type).alias(canonical_column)
                )
            else:
                select_expressions.append(
                    F.lit(None).cast(target_type).alias(canonical_column)
                )

    df_reconstructed = df.select(*select_expressions)

    df_reconstructed = (
        df_reconstructed
        .withColumn("process_id", F.lit(PROCESS_ID))
        .withColumn("recovery_category", F.lit(problem_category))
    )

    return df_reconstructed


## 4. Localización de archivos problemáticos

Se analizan los archivos que estén en:

`data/raw/bad_parquet/`

Si la carpeta está vacía, el notebook no falla. Simplemente genera una tabla de cuarentena vacía con la estructura correcta.


In [39]:
# Buscar archivos problemáticos.

bad_files = []

if BAD_PARQUET_PATH.exists():
    bad_files = [
        file
        for file in BAD_PARQUET_PATH.rglob("*")
        if file.is_file()
    ]

print("Archivos encontrados en data/raw/bad_parquet:", len(bad_files))

for file in bad_files:
    print(file)


Archivos encontrados en data/raw/bad_parquet: 5
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet\ARROW-GH-41317.parquet
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet\ARROW-GH-41321.parquet
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet\ARROW-GH-45185.parquet
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet\ARROW-RS-GH-6229-DICTHEADER.parquet
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet\PARQUET-1481.parquet


## 5. Procesamiento individual de archivos problemáticos

Cada archivo se procesa de forma independiente.  
Esto evita que un archivo dañado detenga toda la ejecución.

Para cada archivo se realiza:

1. Validación de tamaño.
2. Validación de formato.
3. Lectura controlada con Spark.
4. Diagnóstico de esquema.
5. Clasificación del problema.
6. Intento de reconstrucción si es recuperable.
7. Registro de cuarentena con evidencia técnica.


In [40]:
# Procesamiento individual de archivos problemáticos.

quarantine_records = []
recovered_dfs = []

for file_path in bad_files:
    file_name = file_path.name
    file_size_mb = round(file_path.stat().st_size / (1024 * 1024), 6)

    print("=" * 100)
    print("Analizando archivo:", file_name)

    df_test = None

    if file_size_mb == 0:
        problem_category = NOT_RECOVERABLE_EMPTY_FILE
        recoverability = get_recoverability(problem_category)
        error_type = get_error_type(problem_category)

        quarantine_records.append({
            "process_id": PROCESS_ID,
            "file_name": file_name,
            "file_path": str(file_path),
            "file_size_mb": file_size_mb,
            "problem_category": problem_category,
            "recoverability": recoverability,
            "error_type": error_type,
            "read_status": "NOT_RECOVERABLE",
            "reconstruction_status": "NOT_ATTEMPTED",
            "record_count": None,
            "column_count": None,
            "missing_columns": json.dumps(CANONICAL_COLUMNS, ensure_ascii=False),
            "additional_columns": json.dumps([], ensure_ascii=False),
            "incompatible_types": json.dumps([], ensure_ascii=False),
            "exception_message": "Archivo vacío detectado antes de lectura Spark.",
            "failed_stage": "FASE_03_PREVALIDACION_ARCHIVO",
            "rejection_reason": "Archivo vacío.",
            "recommended_action": "Solicitar nuevamente el archivo al origen o excluirlo del procesamiento productivo.",
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        print("Estado: NOT_RECOVERABLE")
        print("Categoría:", problem_category)
        continue

    if not file_name.lower().endswith(".parquet"):
        problem_category = NOT_RECOVERABLE_UNSUPPORTED_FORMAT
        recoverability = get_recoverability(problem_category)
        error_type = get_error_type(problem_category)

        quarantine_records.append({
            "process_id": PROCESS_ID,
            "file_name": file_name,
            "file_path": str(file_path),
            "file_size_mb": file_size_mb,
            "problem_category": problem_category,
            "recoverability": recoverability,
            "error_type": error_type,
            "read_status": "NOT_RECOVERABLE",
            "reconstruction_status": "NOT_ATTEMPTED",
            "record_count": None,
            "column_count": None,
            "missing_columns": None,
            "additional_columns": None,
            "incompatible_types": None,
            "exception_message": "Formato no soportado. El archivo no tiene extensión .parquet.",
            "failed_stage": "FASE_03_PREVALIDACION_FORMATO",
            "rejection_reason": "Formato no soportado.",
            "recommended_action": "Verificar el formato original y convertirlo correctamente a Parquet antes de procesarlo.",
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        print("Estado: NOT_RECOVERABLE")
        print("Categoría:", problem_category)
        continue

    try:
        df_test = spark.read.parquet(spark_path(file_path))

        record_count = df_test.count()
        column_count = len(df_test.columns)

        problem_category, missing_columns, additional_columns, incompatible_types = classify_readable_file(
            df_test,
            record_count
        )

        recoverability = get_recoverability(problem_category)
        error_type = get_error_type(problem_category)

        exception_message = None
        failed_stage = None

        if record_count == 0:
            read_status = "NOT_RECOVERABLE"
            reconstruction_status = "NOT_ATTEMPTED"
            failed_stage = "FASE_03_ARCHIVO_SIN_REGISTROS"
            recommended_action = "Archivo sin registros útiles. Mantener en cuarentena y solicitar revisión del archivo origen."
        else:
            read_status = "READABLE"
            reconstruction_status = "NOT_ATTEMPTED"
            recommended_action = "Revisar archivo y validar si puede integrarse al flujo principal."

            try:
                if recoverability in [RECOVERABLE, PARTIALLY_RECOVERABLE_STATUS]:
                    recovered_df = reconstruct_to_canonical(df_test, file_path, problem_category)
                    recovered_df.limit(1).count()
                    recovered_dfs.append(recovered_df)
                    reconstruction_status = "RECONSTRUCTED_TO_CANONICAL_SCHEMA"
                    recommended_action = "Archivo reconstruido hacia esquema canónico y almacenado en bronze/recovered_files."

            except Exception as reconstruction_error:
                reconstruction_status = "RECONSTRUCTION_FAILED"
                exception_message = str(reconstruction_error)[:1500]
                failed_stage = "FASE_03_RECONSTRUCCION_CANONICA"
                recommended_action = "Mantener archivo en cuarentena y revisar reglas de casteo u homologación."

        quarantine_records.append({
            "process_id": PROCESS_ID,
            "file_name": file_name,
            "file_path": str(file_path),
            "file_size_mb": file_size_mb,
            "problem_category": problem_category,
            "recoverability": recoverability,
            "error_type": error_type,
            "read_status": read_status,
            "reconstruction_status": reconstruction_status,
            "record_count": int(record_count),
            "column_count": int(column_count),
            "missing_columns": json.dumps(missing_columns, ensure_ascii=False),
            "additional_columns": json.dumps(additional_columns, ensure_ascii=False),
            "incompatible_types": json.dumps(incompatible_types, ensure_ascii=False),
            "exception_message": exception_message,
            "failed_stage": failed_stage,
            "rejection_reason": problem_category,
            "recommended_action": recommended_action,
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        print("Estado:", read_status)
        print("Categoría:", problem_category)
        print("Recuperabilidad:", recoverability)
        print("Tipo de error:", error_type)
        print("Reconstrucción:", reconstruction_status)
        print("Registros:", record_count)
        print("Columnas:", column_count)

    except Exception as error:
        exception_message = str(error)

        if df_test is not None:
            problem_category = PARTIALLY_RECOVERABLE
            read_status = "PARTIALLY_RECOVERABLE"
            reconstruction_status = "RECONSTRUCTION_FAILED"
            failed_stage = "FASE_03_MATERIALIZACION_PARQUET"
            recommended_action = (
                "Revisar manualmente. El archivo permite lectura parcial de esquema, "
                "pero falla al materializar registros o completar la lectura."
            )

            try:
                column_count = len(df_test.columns)
                missing_columns, additional_columns, incompatible_types = diagnose_schema(df_test)
            except Exception:
                column_count = None
                missing_columns = None
                additional_columns = None
                incompatible_types = None

        else:
            problem_category = classify_exception(file_path, exception_message)
            read_status = "NOT_RECOVERABLE"
            reconstruction_status = "NOT_ATTEMPTED"
            failed_stage = "FASE_03_LECTURA_PARQUET"
            recommended_action = (
                "No cargar al pipeline principal. Mantener evidencia en cuarentena "
                "y revisar archivo origen."
            )
            column_count = None
            missing_columns = None
            additional_columns = None
            incompatible_types = None

        recoverability = get_recoverability(problem_category)
        error_type = get_error_type(problem_category)

        quarantine_records.append({
            "process_id": PROCESS_ID,
            "file_name": file_name,
            "file_path": str(file_path),
            "file_size_mb": file_size_mb,
            "problem_category": problem_category,
            "recoverability": recoverability,
            "error_type": error_type,
            "read_status": read_status,
            "reconstruction_status": reconstruction_status,
            "record_count": None,
            "column_count": column_count,
            "missing_columns": json.dumps(missing_columns, ensure_ascii=False) if missing_columns is not None else None,
            "additional_columns": json.dumps(additional_columns, ensure_ascii=False) if additional_columns is not None else None,
            "incompatible_types": json.dumps(incompatible_types, ensure_ascii=False) if incompatible_types is not None else None,
            "exception_message": exception_message[:1500],
            "failed_stage": failed_stage,
            "rejection_reason": problem_category,
            "recommended_action": recommended_action,
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        print("Estado:", read_status)
        print("Categoría:", problem_category)
        print("Recuperabilidad:", recoverability)
        print("Tipo de error:", error_type)
        print("Reconstrucción:", reconstruction_status)
        print("Error capturado:", exception_message[:300])

print("=" * 100)
print("Archivos procesados:", len(bad_files))
print("Registros de cuarentena generados:", len(quarantine_records))
print("DataFrames recuperados:", len(recovered_dfs))

Analizando archivo: ARROW-GH-41317.parquet
Estado: NOT_RECOVERABLE
Categoría: NOT_RECOVERABLE_CORRUPT_METADATA
Recuperabilidad: NOT_RECOVERABLE
Tipo de error: PHYSICAL_FILE_ERROR
Reconstrucción: NOT_ATTEMPTED
Error capturado: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846
Analizando archivo: ARROW-GH-41321.parquet
Estado: NOT_RECOVERABLE
Categoría: NOT_RECOVERABLE_CORRUPT_METADATA
Recuperabilidad: NOT_RECOVERABLE
Tipo de error: PHYSICAL_FILE_ERROR
Reconstrucción: NOT_ATTEMPTED
Error capturado: [PARQUET_TYPE_ILLEGAL] Illegal Parquet type: INT32 (TIME(MILLIS,true)). SQLSTATE: 42846
Analizando archivo: ARROW-GH-45185.parquet
Estado: READABLE
Categoría: RECUPERABLE_MISSING_COLUMNS
Recuperabilidad: RECOVERABLE
Tipo de error: SCHEMA_ERROR
Reconstrucción: RECONSTRUCTED_TO_CANONICAL_SCHEMA
Registros: 5
Columnas: 1
Analizando archivo: ARROW-RS-GH-6229-DICTHEADER.parquet
Estado: NOT_RECOVERABLE
Categoría: NOT_RECOVERABLE_EMPTY_FILE
Recuperabilidad: NOT_RE

## 6. Construcción de la tabla de cuarentena

La cuarentena contiene la evidencia solicitada por el enunciado:

- Archivo original o referencia al archivo.
- Motivo de rechazo.
- Excepción capturada.
- Fecha de procesamiento.
- Etapa donde falló.
- Acción recomendada.

También se agregan campos de apoyo para auditoría: recuperabilidad, tipo de error, estado de lectura y estado de reconstrucción.


In [41]:
# Crear DataFrame de cuarentena con esquema explícito.

quarantine_schema = StructType([
    StructField("process_id", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_size_mb", DoubleType(), True),
    StructField("problem_category", StringType(), True),
    StructField("recoverability", StringType(), True),
    StructField("error_type", StringType(), True),
    StructField("read_status", StringType(), True),
    StructField("reconstruction_status", StringType(), True),
    StructField("record_count", IntegerType(), True),
    StructField("column_count", IntegerType(), True),
    StructField("missing_columns", StringType(), True),
    StructField("additional_columns", StringType(), True),
    StructField("incompatible_types", StringType(), True),
    StructField("exception_message", StringType(), True),
    StructField("failed_stage", StringType(), True),
    StructField("rejection_reason", StringType(), True),
    StructField("recommended_action", StringType(), True),
    StructField("processed_at", StringType(), True)
])

quarantine_df = spark.createDataFrame(quarantine_records, schema=quarantine_schema)

print("Tabla de cuarentena generada:")
quarantine_df.show(truncate=False)


Tabla de cuarentena generada:
+---------------+-----------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------+---------------------+-------------------+---------------------+---------------------------------+------------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [42]:
# Guardar la tabla de cuarentena en formato Parquet.

quarantine_output_path = QUARANTINE_PATH / f"parquet_file_quarantine_{PROCESS_ID}"

(
    quarantine_df
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(spark_path(quarantine_output_path))
)

print("Tabla de cuarentena guardada en:")
print(quarantine_output_path)


Tabla de cuarentena guardada en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\parquet_file_quarantine_20260617_171909


## 7. Copia de evidencia física

Además de guardar la referencia en la tabla de cuarentena, se copia una evidencia física de los archivos analizados hacia `data/quarantine/evidence_files_<process_id>`.

Esto ayuda a demostrar trazabilidad y facilita la revisión posterior.


In [43]:

# Copiar archivos problemáticos como evidencia física.

evidence_path = QUARANTINE_PATH / f"evidence_files_{PROCESS_ID}"
evidence_path.mkdir(parents=True, exist_ok=True)

copied_files = 0

for file_path in bad_files:
    destination = evidence_path / file_path.name

    try:
        shutil.copy2(file_path, destination)
        copied_files += 1
        print("Archivo copiado como evidencia:", destination)
    except Exception as error:
        print("No se pudo copiar:", file_path)
        print("Error:", str(error)[:300])

print("Total de archivos copiados como evidencia:", copied_files)
print("Ruta de evidencias:")
print(evidence_path)

Archivo copiado como evidencia: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\evidence_files_20260617_171909\ARROW-GH-41317.parquet
Archivo copiado como evidencia: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\evidence_files_20260617_171909\ARROW-GH-41321.parquet
Archivo copiado como evidencia: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\evidence_files_20260617_171909\ARROW-GH-45185.parquet
Archivo copiado como evidencia: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\evidence_files_20260617_171909\ARROW-RS-GH-6229-DICTHEADER.parquet
Archivo copiado como evidencia: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\evidence_files_20260617_171909\PARQUET-1481.parquet
Total de archivos copiados como evidencia: 5
Ruta de evidencias:
c:\Users\

## 8. Escritura de archivos recuperados

Si un archivo fue clasificado como recuperable o parcialmente recuperable, el notebook intenta reconstruirlo al esquema canónico y lo guarda en:

`data/bronze/recovered_files/`

Esto demuestra que la fase no solo detecta problemas, sino que también intenta recuperar información útil.


In [44]:
# Unir y guardar los DataFrames recuperados, si existen.

recovered_output_path = BRONZE_PATH / "recovered_files" / f"process_id={PROCESS_ID}"

if recovered_dfs:
    recovered_canonical_df = recovered_dfs[0]

    for df in recovered_dfs[1:]:
        recovered_canonical_df = recovered_canonical_df.unionByName(df, allowMissingColumns=True)

    (
        recovered_canonical_df
        .repartition(4)
        .write
        .mode("overwrite")
        .parquet(spark_path(recovered_output_path))
    )

    print("Archivos recuperados guardados en:")
    print(recovered_output_path)

    recovered_canonical_df.select(
        "trip_id",
        "service_type",
        "source_file",
        "quality_status",
        "process_id",
        "recovery_category"
    ).show(10, truncate=False)

else:
    recovered_canonical_df = None
    print("No existen archivos recuperables para guardar en bronze/recovered_files.")


Archivos recuperados guardados en:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\bronze\recovered_files\process_id=20260617_171909
+----------------------------------------------------------------+------------+----------------------+-------------------------------------+---------------+---------------------------+
|trip_id                                                         |service_type|source_file           |quality_status                       |process_id     |recovery_category          |
+----------------------------------------------------------------+------------+----------------------+-------------------------------------+---------------+---------------------------+
|08c868df75ee96c689b02ae36e6614eb896e2b029758a1b84556833a88733aca|unknown     |ARROW-GH-45185.parquet|RECOVERED_RECUPERABLE_MISSING_COLUMNS|20260617_171909|RECUPERABLE_MISSING_COLUMNS|
|586e905662d413eb06afedda446c6af6dee6bcab0d22cfc37c18cb1229e0768e|unknown     |ARROW-GH-4

## 9. Resumen de clasificación

Este resumen permite ver cuántos archivos quedaron en cada categoría y si fueron recuperados o enviados a cuarentena.


In [45]:
# Resumen de archivos por categoría.

summary_df = (
    quarantine_df
    .groupBy("problem_category", "recoverability", "error_type", "read_status", "reconstruction_status")
    .agg(
        F.count("*").alias("total_files"),
        F.round(F.sum("file_size_mb"), 6).alias("total_size_mb")
    )
    .orderBy("problem_category", "read_status")
)

summary_df.show(truncate=False)


+--------------------------------+---------------------+-------------------+---------------------+---------------------------------+-----------+-------------+
|problem_category                |recoverability       |error_type         |read_status          |reconstruction_status            |total_files|total_size_mb|
+--------------------------------+---------------------+-------------------+---------------------+---------------------------------+-----------+-------------+
|NOT_RECOVERABLE_CORRUPT_METADATA|NOT_RECOVERABLE      |PHYSICAL_FILE_ERROR|NOT_RECOVERABLE      |NOT_ATTEMPTED                    |2          |0.139226     |
|NOT_RECOVERABLE_EMPTY_FILE      |NOT_RECOVERABLE      |PHYSICAL_FILE_ERROR|NOT_RECOVERABLE      |NOT_ATTEMPTED                    |1          |5.08E-4      |
|PARTIALLY_RECOVERABLE           |PARTIALLY_RECOVERABLE|BUSINESS_RULE_ERROR|PARTIALLY_RECOVERABLE|RECONSTRUCTION_FAILED            |1          |4.3E-4       |
|RECUPERABLE_MISSING_COLUMNS     |RECOVERABLE 

In [46]:
# Validación de categorías obligatorias.

mandatory_categories = [
    RECUPERABLE_SCHEMA_MISMATCH,
    RECUPERABLE_MISSING_COLUMNS,
    RECUPERABLE_TYPE_CASTING,
    PARTIALLY_RECOVERABLE,
    NOT_RECOVERABLE_CORRUPT_METADATA,
    NOT_RECOVERABLE_EMPTY_FILE,
    NOT_RECOVERABLE_UNSUPPORTED_FORMAT
]

print("Categorías obligatorias contempladas por el notebook:")
for category in mandatory_categories:
    print("-", category)

print("Validación final: todas las categorías obligatorias están definidas en la lógica del notebook.")


Categorías obligatorias contempladas por el notebook:
- RECUPERABLE_SCHEMA_MISMATCH
- RECUPERABLE_MISSING_COLUMNS
- RECUPERABLE_TYPE_CASTING
- PARTIALLY_RECOVERABLE
- NOT_RECOVERABLE_CORRUPT_METADATA
- NOT_RECOVERABLE_EMPTY_FILE
- NOT_RECOVERABLE_UNSUPPORTED_FORMAT
Validación final: todas las categorías obligatorias están definidas en la lógica del notebook.


## 10. Interpretación técnica de resultados

La tabla de cuarentena debe interpretarse así:

- `problem_category`: categoría obligatoria asignada al archivo.
- `recoverability`: indica si el archivo es recuperable, parcialmente recuperable o no recuperable.
- `error_type`: indica el tipo general del problema.
- `read_status`: indica si Spark pudo leer el archivo.
- `reconstruction_status`: indica si se intentó y logró reconstruirlo.
- `exception_message`: guarda la excepción técnica capturada.
- `failed_stage`: indica la etapa donde se produjo el problema.
- `recommended_action`: explica qué se recomienda hacer con ese archivo.

Esta evidencia permite auditar el tratamiento dado a cada archivo problemático.


In [47]:
# Mostrar columnas principales de la tabla de cuarentena para revisión.

quarantine_df.select(
    "file_name",
    "problem_category",
    "recoverability",
    "error_type",
    "read_status",
    "reconstruction_status",
    "failed_stage",
    "recommended_action"
).show(truncate=False)


+-----------------------------------+--------------------------------+---------------------+-------------------+---------------------+---------------------------------+-------------------------------+--------------------------------------------------------------------------------------------------------------------------------+
|file_name                          |problem_category                |recoverability       |error_type         |read_status          |reconstruction_status            |failed_stage                   |recommended_action                                                                                                              |
+-----------------------------------+--------------------------------+---------------------+-------------------+---------------------+---------------------------------+-------------------------------+--------------------------------------------------------------------------------------------------------------------------------+
|ARROW-GH-

## Conclusión de la Fase 3

En esta fase se diseñó e implementó una estrategia profesional para tratar archivos Parquet dañados o parcialmente ilegibles.

El proceso analiza cada archivo de forma individual, captura excepciones técnicas, clasifica el problema en una categoría obligatoria, diferencia si el archivo es recuperable, parcialmente recuperable o no recuperable, registra evidencia en la capa de cuarentena y, cuando es posible, reconstruye el archivo hacia el esquema canónico.

Los archivos no recuperables no se ignoran: quedan documentados en `data/quarantine/` con su referencia, motivo, excepción, etapa de falla, fecha de procesamiento y acción recomendada.

Los archivos recuperables se reconstruyen y se almacenan en `data/bronze/recovered_files/`, manteniendo trazabilidad mediante `process_id`, `source_file`, `quality_status` y `recovery_category`.

Con esto se demuestra que el pipeline puede manejar archivos problemáticos sin detener todo el proceso ETL, cumpliendo el objetivo de la Fase 3.
